# First Implementation of the RSA model 

In [58]:
import numpy as np 
import random as rd
import spacy
import random as rd
import json 
import sys 

sys.path.insert(1, '/home/user/Documents/Sprachgebrauchsmodellierung_RSA/Hausarbeit_RSA/repo/scripts/')
from gamestate import gs
nlp = spacy.load('de_core_news_sm')


In [55]:
class Agent:
    def __init__(self, player, knows_imp=False): 
         self.player = player
         self.role = ROLES[player]
         self.target_imp = self.get_target()
         self.alpha = RATIONALITY[player]
         if knows_imp: 
            self.priors = {}
            for p in PLAYERS:
                if p==self.player:
                    self.priors[p]= self.get_priors()
                elif ROLES[p]=="agent":
                    self.priors[p]= self.get_priors(knows_imp=False)
                else:
                    self.priors[p]= self.get_priors(knows_t=False)


    def get_target(self):
        mv = max(FREQ.values())
        res = [k for k in FREQ if FREQ[k] == mv]
        return(rd.choice(res))
    
    def get_priors(self, knows_t = True, knows_imp = True):
        priors = []
        # P1 weiß wer der Imposter ist 
        for p, w in W: 
            if knows_imp: 
                if ROLES[p]=="agent":
                    priors.append(0)
                else:
                    if knows_t: 
                        if w==TARGET:
                            priors.append(1)
                        else:
                            priors.append(0)
                    else:
                        priors.append(FREQ[w])
            else:
                if p==self.player:
                    priors.append(0)
                else:
                    priors.append(1/(len(PLAYERS)-1))
                    
        priors = [w/sum(priors) for w in priors]
        priors = np.array(priors)
        priors.shape = (len(PLAYERS),len(V))
        return(priors)
    
    def get_P_u_w(u,w,t_imp,speaker="P1"):
        imp_ind, t_ind = w
        if PLAYERS[imp_ind]==speaker: 
            P_u_w = SIMS[t_ind, u]
        else:
            P_u_w = SIMS[t_imp, u]
        return(P_u_w)

    def L0(self, u, w, listener="P2"):
        imp, t = w 
        imp = PLAYERS.index(imp)
        priors_l = self.priors[listener]
        t = V.index(t)
        u = V.index(u)
        P_w_u = get_P_u_w(u,(imp,t),target_imp_ind) * priors_l[imp][t]
        SUM = np.nansum([get_P_u_w(u,w_n,target_imp_ind) * priors_l[w_n[0],w_n[1]] for w_n in range(len(W))])
        P_w_u = P_w_u/SUM
        return(P_w_u)
    
    def utility(self, u, w, partner="P2", imp="P3"):
        #print(f"listen(u,w,listener=partner): {listen(u,w,listener=partner)}, listen(u,(P3,t),listener=imp): {listen(u,(imp,w[1]),listener=imp)}")
        # Anwendung von log-Funktion auf Differenz beruht ausschließlich auf ausprobieren
        U = np.log(self.L0(u,w,listener=partner)) + np.log(self.L0(u,(imp,w[1]),listener=imp)) # weil auf logarithmischer Skala negativ 
        #SUM = np.nansum([np.log(listen(u_n,w,listener=partner)) - np.log(listen(u_n,w,listener=imp)) for u_n in range(len(V))])
        return(U)
    
    def S1(self, u,w, alpha=1): 
        P_u_w = np.exp(alpha * utility(u,w))
        SUM = np.nansum([np.exp(alpha * utility(u_n, w)) for u_n in V])
        P_u_w = P_u_w/SUM
        return(P_u_w)
    
    def L1(self, u,w,alpha=1,speaker="P1", listener="P2"):
        imp, t = w
        imp_ind = PLAYERS.index(imp)
        t_ind = PLAYERS.index(t)
        if imp=="speaker":
            P_w_u = S1(u,w,alpha=alpha) * self.priors[listener][imp_ind][t_ind]
        else:
            P_w_u = S1(u,w,alpha=alpha) * self.priors[listener][imp_ind][t_ind]
        SUM = np.nansum([S1(u,w_n,alpha=alpha) * self.priors[listener][PLAYERS.index(w_n[0])][V.index(w_n[1])] for w_n in W])
        P_w_u = P_w_u/SUM
        return(P_w_u)



## next steps 

- implement cosine similarity 
- think about how to adjust probabilities with L1 functions 


In [33]:
with open('../output/random_nouns.json', 'r') as file:
    rd_100 = json.load(file)

print(rd_100)


['Ball', 'Künstler', 'Spaziergang', 'Partnerin', 'Bayer', 'Ereignis', 'Feuerwehr', 'Petition', 'Bundestag', 'Erfüllung', 'Ausweis', 'Rückkehr', 'Zitat', 'Verfasst', 'Schülerin', 'Wette', 'Aluminium', 'Saal', 'Einbezog', 'Eign', 'Klinik', 'Playstation', 'Nutze', 'Zufriedenheit', 'Opposition', 'Schweine', 'Messung', 'Steigerung', 'Hafen', 'Service', 'Zwischenzeit', 'Kenntnis', 'Rest', 'Terrorismus', 'Ulm', 'Wettbewerb', 'Freund', 'Element', 'Anwohner', 'Bgb', 'Empfänger', 'Rufe', 'Station', 'Milde', 'Gefasst', 'Invasion', 'Herauszufinde', 'Gewalt', 'Realschule', 'Solidarität', 'Produkt', 'Koffer', 'Mann', 'Verbrechen', 'Weisheit', 'Kontinent', 'Komplex', 'Pressemitteilung', 'Vorteil', 'Doktor', 'Symbol', 'Tasse', 'Scheidung', 'Stürmer', 'Statistik', 'Christ', 'Rundschau', 'Furcht', 'Dialog', 'Partie', 'Nation', 'Haus', 'Jungfrau', 'Tiger', 'Zieh', 'Rufe', 'Hochschule', 'Vorwurf', 'Rechtsprechung', 'Hotel', 'Fahrt', 'Wahrheit', 'Meinung', 'Verkehrt', 'Zwischenzeitlich', 'Einleitung', 'Nac

In [ ]:
class Imposter:
    def __init__(self, player): 
         self.player = player
         self.role = roles[player]
         self.target = self.get_target()
         self.P_t = freq.copy()
         self.alpha = rationality[player]
         self.target_imp = self.get_target()

    def get_target(self):
        mv = max(freq.values())
        res = [k for k in freq if freq[k] == mv]
        return(rd.choice(res))
    
    
    def L0(self, u, t, p_player = [2/3, 1/3]):
        p_agent = get_similarity(u,t) * p_player[0]
        p_imposter = get_similarity(u,self.target_imp) * p_player[1]
        p_agent_norm = p_agent/(p_agent + p_imposter)
        p_imposter_norm = p_imposter/(p_agent + p_imposter)
        return((p_agent_norm, p_imposter_norm))

    def S1(self, t):
        p_distr = []
        for u in V: 
            # what exactly is the world? The persona the player wants to communicate? That would always be agent.
            p_u = np.exp(self.alpha * (np.log(self.L0(u, t)[0])- C[u]))
            #print("p_u:", p_u)
            p_distr.append(p_u)

        p_sum = sum(p_distr)
        u_dic = {u: p/p_sum for u,p in zip(V,p_distr)}
        #print(p_distr)
        #print(u_dic)
        return(u_dic)

    def choose_u(self): 
        P_u = self.S1(self.target)
        mv = max(P_u.values())
        res = [k for k in P_u if P_u[k] == mv]
        return(rd.choice(res))
    

    def L1(self, u):
        t_distr = []
        for t in V: 
            p = self.S1(t)[u] * self.P_t[t]
            t_distr.append(p)

        p_sum = sum(t_distr)
        t_dic = {t: p/p_sum for t,p in zip(V,t_distr)}

        # is the new probability distribution a mix of the old distribution and the newly calculated one or is it just the new one? 
        self.P_t = t_dic

        return(t_dic)
    



class GameState:
    def __init__(self): 
        self.previous_words = []

    def say_word(self, word):
        self.previous_words += word